## 


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/03_workflow.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Building an AI Course Assistant with Ollama</h2>
            <h4 style="color:#00bf00;">© 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">In this lab, we build a conversational AI assistant for the course. We use the course summary and syllabus to create a system prompt, send chat messages to an Ollama model through the OpenAI-compatible API, and present the assistant in a Gradio chat interface.

Set `OLLAMA_BASE_URL` in the project-root `.env` file. The model name is configured in the notebook.
            </span>
        </td>
    </tr>
</table>

In [9]:
from dotenv import load_dotenv
import os

from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [10]:
load_dotenv(override=True)
MODEL = "gpt-oss:120b"
# Always remember to do this!
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

openai = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [11]:
reader = PdfReader("docs/syllabus.pdf")
syllabus = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        syllabus += text

In [ ]:
print(syllabus)

In [12]:
with open("docs/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Elena"}
]

In [ ]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Elena"},
    {"role": "assistant", "content": "Well hi there, Elena. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

In [13]:
system_prompt = f"""

# Your role

You are a digital course assistant running on a website, chatting with visitors of the website.
You represent the course website you are on.
You answer questions related to their course, background, skills and experience.

Here are the details of the course you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital assistan of this course.

# Context

Here is a summary of the course so that you can answer questions:

{syllabus}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential student across the website.
Avoid answering questions that are not related to the course , background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital assistant of the course you are representing. Represent the course.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
display(Markdown(response.choices[0].message.content))

In [14]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# And now - TOOLS!

Let's start with a function...

In [15]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [25]:
record_email_tool("test@test.com")

Tool called to record an email: test@test.com


'Email received'

## Step 1 - write some json to describe the tool


In [17]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [18]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

In [35]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [36]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: leviio@lpokp.com


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [38]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: leviio@lpokp.com
Tool called to record an email: hjkhkuh@jiji.com


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!